Pada notebook ini akan melakukan proses prediksi data gaji menggunakan dataset yang diberikan oleh SanberCode yang berisikan variabel parameter dalam datasetnya yaitu kolom Umur, Kelas Pekerja, Berat Akhir, Pendidikan, Jmlh Tahun Pendidikan, Status Perkawinan, Pekerjaan, Jenis Kelamin, Keuntungan Kapital, Kerugian Capital, Jam per Minggu dan Gaji serta pada dataset test dihilangkan kolom gaji untuk melakukan prediksi atau untuk mendapatkan output prediksi dari proses Machine Learning nya. Model Algoritma Machine Learning yang digunakan pada pengujian ini yaitu *Algoritma Decision Tree dan Random Forest* yang dilakukan *hyperpameter tuning menggunakan GridSearchCV* untuk mendapatkan hyperpamareter terbaik. Sebelum melakukan fitting kedalam model algoritma nya dilakukan preprocessing atau cleaning dan normalisasi dataset agar dapat lebih mudah melakukan proses machine learning dan mengahasilkan akurasi yang lebih akurat. Hasil akhir pada Notebook ini barupa label output pada kolom gaji hasil prediksi model algoritma Machine Learning, untuk akurasinya didapatkan setelah submit ke competition di Kaggle dan didapatkan **Model algoritma Random Forest memberikan hasil akurasi tebaik.**

# Importing Libraries

Pada proses ini saya melakukan import library atau package yang akan digunakan 
* Pandas -> untuk meproses data 
* LabelEncoder dan np -> untuk mengubah data object/string menjadi int dan memproses angka 
* plt dan sn -> untuk visualisasi
* StandardScaler -> untuk normalisasi data angka
* RandomForestClassifier dan DecisionTreeClassifier -> untuk mengimport model algoritma machine learning
* GridSearchCV -> untuk hyperparameter tuning


In [ ]:
# import library
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
import seaborn as sn
import matplotlib.pyplot as plt

# Import csv files using pandas

Memasukkan dataset yang tersedia yang terbagi mendaji data latih/data train yang disimpan dalam variable *train* dan data uji/data test yang disimpan dalam variable *test* yang merupakan data dengan ekstensi csv menggunakan pandas lalu menampilkan 10 data teratas pada dataset tersebut.

In [ ]:
# read data
train = pd.read_csv('../input/sanber1/train.csv')
train.head(10)

In [ ]:
test = pd.read_csv('../input/sanber1/test.csv')
test.head(10)

# Checking dataset info (Non-Null Count and Data Type)

Pada proses ini dilakukan proses untuk memeriksa deskripsi kolom berupa nama kolom dan memerisa kolom yang kosong (bila ada) dan tipe data nya. 

Dan didapatkan :
* Terdapat 35994 data pada data train dan 9599 data pada data uji.
* Tidak ada data yang null/kosong.
* Pada data train terdapat 7 data integer dan 6 data object dan Pada data test terdapat 3 data float, 4 data integer dan 5 data object.
* Terdapat 13 kolom pada data train dan 12 kolom pada data test.
* Pada tiap-tiap dataset berisikan kolom Umur, Kelas Pekerja, Berat Akhir, Pendidikan, Jmlh Tahun Pendidikan, Status Perkawinan, Pekerjaan, Jenis Kelamin, Keuntungan Kapital, Kerugian Capital, Jam per Minggu dan Gaji dan pada data test.


In [ ]:
# Check info data
train.info()

test.info()

# Replacing '?' with 'Tidak Diketahui'

Pada dataset ini ternyata terdapat data yang berisikan data '?' yang berarti data tersebut tidak diketahui isinya, untuk merapihkan data ini saya memilih untuk mengganti '?' dengan teks 'tidak diketahui' daripada menghapus kolom yang berisikan '?'.

In [ ]:
# replace test dataset
test = test.replace({'?': 'Tidak Deketahui'})

In [ ]:
test

In [ ]:
# replace train dataset
train = train.replace({'?': 'Tidak Deketahui'})

In [ ]:
train

# Encode/Transform string data type using .replace

Pada proses ini saya mengubah data yang sebelumnya merupakan object menjadi data angka seperti pada kolom gaji yang memiliki gaji <= 6 juta menjadi 0 dan gaji > 6 juta menjadi 1 serta untuk kolom jenis kelamin perempuan menjadi 0 dan laki-laki menjadi 1 menggunakan replace sedangkan untuk data pada kolom kelas pekerja, pendidikan, pekerjaan dan status perkawinan menggunakan encoder yaitu labelEncoder untuk melakukan encode datanya untuk memudahkan proses machine learning.

In [ ]:
# encode column gaji
obj_gaji = {'<= 6 juta':0, '> 6 juta':1}
train['Gaji'] = train['Gaji'].replace(obj_gaji)

# encode column jenis kelamin
obj_jk = {'Perempuan':0, 'Laki2':1}
train['Jenis Kelamin'] = train['Jenis Kelamin'].replace(obj_jk)
test['Jenis Kelamin'] = test['Jenis Kelamin'].replace(obj_jk)

Encode/Transform String Data type using LabelEncoder

In [ ]:
# call LabelEncoder and save in variable 'le'
le = LabelEncoder()

# fit 'le' to 'Kelas Pekerja'
le.fit(train['Kelas Pekerja'])
le.fit(test['Kelas Pekerja'])

# list of unique rows in 'Kelas Pekerja' column
list(le.classes_)

In [ ]:
# transform data 
train['Kelas Pekerja']=le.transform(train['Kelas Pekerja'])
test['Kelas Pekerja']=le.transform(test['Kelas Pekerja'])

print(np.sort(test['Kelas Pekerja'].unique()))

In [ ]:
# same step
le.fit(train['Pendidikan'])
le.fit(test['Pendidikan'])
list(le.classes_)

In [ ]:
train['Pendidikan']=le.transform(train['Pendidikan'])
test['Pendidikan']=le.transform(test['Pendidikan'])

print(np.sort(test['Pendidikan'].unique()))

In [ ]:
le.fit(train['Pekerjaan'])
le.fit(test['Pekerjaan'])
list(le.classes_)

In [ ]:
train['Pekerjaan']=le.transform(train['Pekerjaan'])
test['Pekerjaan']=le.transform(test['Pekerjaan'])

print(np.sort(test['Pekerjaan'].unique()))

In [ ]:
le.fit(train['Status Perkawinan'])
le.fit(test['Status Perkawinan'])
list(le.classes_)

In [ ]:
train['Status Perkawinan']=le.transform(train['Status Perkawinan'])
test['Status Perkawinan']=le.transform(test['Status Perkawinan'])

# check numbers after transforming data
print(np.sort(test['Status Perkawinan'].unique()))

In [ ]:
# Recheck if our columns are all in Int/Float dtypes
test.info()
train.info()

> Akhir proses ini mengahsilkan semua data sudah berupa tipe data integer.

# Visualization

Pada proses ini saya melakukan penggambaran atau visualisasi data pada axis x yaitu ada data kelas pekerja, jumlah tahun pendidikan, dan jenis kelamin serta pada axis y ayaitu gaji yang digambarkan dengan menggunakan grouping menggunakan data mean.

In [ ]:
print('Kelas Pekerja')
x = train[['Kelas Pekerja','Gaji']].groupby(['Kelas Pekerja'], as_index=False).mean()['Kelas Pekerja']
y = train[['Kelas Pekerja','Gaji']].groupby(['Kelas Pekerja'], as_index=False).mean()['Gaji']
plt.bar(x,y)
plt.xlabel('Kelas Pekerja')
plt.ylabel('Gaji')
plt.show()

print('Jumlah Tahun Pendidikan')
x = train[['Jmlh Tahun Pendidikan','Gaji']].groupby(['Jmlh Tahun Pendidikan'], as_index=False).mean()['Jmlh Tahun Pendidikan']
y = train[['Jmlh Tahun Pendidikan','Gaji']].groupby(['Jmlh Tahun Pendidikan'], as_index=False).mean()['Gaji']
plt.bar(x,y)
plt.xlabel('Jmlh Tahun Pendidikan')
plt.ylabel('Gaji')
plt.show()

print('Jumlah Tahun Pendidikan')
x = train[['Jenis Kelamin','Gaji']].groupby(['Jenis Kelamin'], as_index=False).mean()['Jenis Kelamin']
y = train[['Jenis Kelamin','Gaji']].groupby(['Jenis Kelamin'], as_index=False).mean()['Gaji']
plt.bar(x,y)
plt.xlabel('Jenis Kelamin')
plt.ylabel('Gaji')
plt.show()

> Gambar 1
Pada Kelas pekerja didapatkan gambaran ukuran rataan gaji berdasarkan kelas pekerja dimana kelas pekerja pada kelas 2 memiliki rataan gaji yang paling besar yaitu 'Pemerintah Negara' diikuti dengan kelas pekerja kelas 4 yaitu 'Pemerintah Provinsi' sedangkan untuk rataan gaji terendah  yaitu pada kelas 7 yaitu bukan pekerja dan kelas 5 yaitu pekerja tanpa bayaran.


> Gambar 2
Pada Jumlah Tahun Pendidikan didapatkan gambaran ukuran rataan gaji berdasarkan jumlah tahun pendidikan dimana terlihat grafik yang linear semakin tinggi jumlah tahun pendidikan semakin besar juga rataan gaji nya.


> Gambar 3
Menggambarkan peserbarah rataan gaji berdasarkan jenis kelamin dimana didapatkan pada bar 1 merupakan gambaran rataan gaji untuk jenis kelamin peremouan lebih rendah daripada rataan gaji pada bar ke-2 untuk laki-laki.


# Correlation Matrix

Pada proses ini saya melakukan penggambaran visualisasi untuk menggambarkan korelasi matriks dari tiap kolom yang ada pada dataset dan saya menggunakan kolom gaji sebagai kolom parameter untuk mengukur relasi dengan keterkaitannya dan untuk memilih kolom yang akan saya gunakan untuk proses machine learning selanjutnya. Didapatkan 2 kolom yang memiliki korelasi tertinggi dengan kolom gaji yaitu kolom status perkawinan dan jumlah tahun pendidikan.

In [ ]:
plt.figure(figsize=(19, 15))
corrMatrix = train.corr()
sn.heatmap(corrMatrix, annot=True)
plt.show()

In [ ]:
drop_elements = ['Berat Akhir', 'Keuntungan Kapital' ,'Kelas Pekerja']
train = train.drop(drop_elements, axis=1)
test = test.drop(drop_elements, axis=1)

In [ ]:
# datatype to int
train = train.astype(int)
test = test.astype(int)

# Normalisasi data 

Pada proses ini melakukan normalisasi menggunakan standard scaler dengan menghapus mean dan menskalakan ke varians unit

In [ ]:
std_scaler = StandardScaler()
data_scale = std_scaler.fit_transform(train)

data_scale

# Choosing X and y (parameters and label)

Pada proses ini saya membagi variable X sebagai data fitur dan variable y sebagai data lebel/output dengan menghapus kolom gaji pada variable X dan menjadikan kolom gaji pada variable y.

In [ ]:
X = train.drop('Gaji', axis=1)
y = train['Gaji']
X_test = test

# DecisionTree

Pada proses ini saya melakukan pengujian menggunakan model algoritma Decision Tree dengan hyperparameter terbaik yaitu {'criterion': 'entropy', 'splitter': 'best'}

In [ ]:
# Setting algorithm model
model_dt = DecisionTreeClassifier()

# Tuning Hypermarameters
param_grid = {
    'criterion':['gini', 'entropy'],
    'splitter':['best', 'random']
}

# Using gscv to processing
gscv = GridSearchCV(model_dt, param_grid=param_grid, scoring='roc_auc', cv=10)

In [ ]:
# fitting model to data
gscv.fit(X,y)

# getting the best hyperparameters 
gscv.best_params_

In [ ]:
# getting score of the best hyperparameters
gscv.best_score_

In [ ]:
# predict using gscv
y_pred = gscv.predict(X_test)

# Save prediction to df_dt dataframe using pandas and show 100 head
df_dt = pd.DataFrame({'id': test['id'], 'Gaji': y_pred})
df_dt.head(100)

# RandomForest

Pada proses ini saya melakukan pengujian menggunakan model algoritma Random Forest dengan hyperparameter terbaik yaitu {'criterion': 'entropy', 'n_estimators': 200}

In [ ]:
model_rf = RandomForestClassifier()
param_grid = {
    'n_estimators':(2,200),
    'criterion':['gini','entropy']    
}
gscv = GridSearchCV(model_rf, param_grid=param_grid, scoring='roc_auc', cv=10)
gscv.fit(X,y)

In [ ]:
gscv.best_params_

In [ ]:
gscv.best_score_

In [ ]:
y_pred = gscv.predict(X_test)

df_rf = pd.DataFrame({'id': test['id'], 'Gaji': y_pred})
df_rf.head(100)